In [0]:
import re
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE  = f"{CATALOG}.bronze"
SILVER  = f"{CATALOG}.silver"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

names = ["FactSalesOrderDetails","FactSalesTerritoryDetails","FactSales",
         "DimReseller","DimDate","DimProduct","DimCustomer"]
raw = {n: spark.table(f"{BRONZE}.{n}") for n in names}
for n, d in raw.items():
    print(f"{n:<28}{d.count():>10,} rows | columns: {d.columns}")

FactSalesOrderDetails          121,253 rows | columns: ['Channel', 'SalesOrderLineKey', 'Sales Order', 'Sales Order Line']
FactSalesTerritoryDetails           11 rows | columns: ['SalesTerritoryKey', 'Region', 'Country', 'Group']
FactSales                      121,253 rows | columns: ['SalesOrderLineKey', 'ResellerKey', 'CustomerKey', 'ProductKey', 'OrderDateKey', 'DueDateKey', 'ShipDateKey', 'SalesTerritoryKey', 'Order Quantity', 'Unit Price', 'Extended Amount', 'Unit Price Discount Pct', 'Product Standard Cost', 'Total Product Cost', 'Sales Amount']
DimReseller                        702 rows | columns: ['ResellerKey', 'Reseller ID', 'Business Type', 'Reseller', 'City', 'State-Province', 'Country-Region', 'Postal Code']
DimDate                          1,461 rows | columns: ['DateKey', 'Date', 'Fiscal Year', 'Fiscal Quarter', 'Month', 'Full Date', 'MonthKey']
DimProduct                         397 rows | columns: ['ProductKey', 'SKU', 'Product', 'Standard Cost', 'Color', 'List Price'

Cleaning Column Names


In [0]:
def clean_cols(df):
    new_cols = [re.sub(r"\s+", "_", c.strip()) for c in df.columns]
    assert len(set(new_cols)) == len(new_cols), "Duplicate column names created!"
    return df.toDF(*new_cols)

t = {n: clean_cols(d) for n, d in raw.items()}
fact_order   = t["FactSalesOrderDetails"]
fact_terr    = t["FactSalesTerritoryDetails"]
fact_sales   = t["FactSales"]
dim_reseller = t["DimReseller"]
dim_date     = t["DimDate"]
dim_product  = t["DimProduct"]
dim_cust     = t["DimCustomer"]
print(fact_sales.columns)

['SalesOrderLineKey', 'ResellerKey', 'CustomerKey', 'ProductKey', 'OrderDateKey', 'DueDateKey', 'ShipDateKey', 'SalesTerritoryKey', 'Order_Quantity', 'Unit_Price', 'Extended_Amount', 'Unit_Price_Discount_Pct', 'Product_Standard_Cost', 'Total_Product_Cost', 'Sales_Amount']


ALL Column Changes

In [0]:
all_changes = []
for name in raw:
    for o, n in zip(raw[name].columns, t[name].columns):
        if o != n:
            all_changes.append((name, o, n))

if all_changes:
    display(spark.createDataFrame(all_changes, ["table", "original_name", "cleaned_name"]))
else:
    print("No columns changed - bronze names already had no spaces")
print(f"Total columns changed: {len(all_changes)}")

table,original_name,cleaned_name
FactSalesOrderDetails,Sales Order,Sales_Order
FactSalesOrderDetails,Sales Order Line,Sales_Order_Line
FactSales,Order Quantity,Order_Quantity
FactSales,Unit Price,Unit_Price
FactSales,Extended Amount,Extended_Amount
FactSales,Unit Price Discount Pct,Unit_Price_Discount_Pct
FactSales,Product Standard Cost,Product_Standard_Cost
FactSales,Total Product Cost,Total_Product_Cost
FactSales,Sales Amount,Sales_Amount
DimReseller,Reseller ID,Reseller_ID


Total columns changed: 19


Convert the following factsales columns to double so that the aggregation can

In [0]:
money_cols = ["Unit_Price", "Extended_Amount", "Product_Standard_Cost",
              "Total_Product_Cost", "Sales_Amount"]

# Bronze stores these as strings like "$2,024.99" -> strip $ and commas, then cast to double
for c in money_cols:
    fact_sales = fact_sales.withColumn(
        c, F.regexp_replace(F.col(c), r"[$,]", "").cast("double")
    )

fact_sales.select(money_cols).printSchema()

# Verify: aggregation works and no value failed to parse (nulls)
display(
    fact_sales.agg(
        *[F.sum(c).alias(f"sum_{c}") for c in money_cols],
        *[F.sum(F.col(c).isNull().cast("int")).alias(f"nulls_{c}") for c in money_cols],
    )
)

root
 |-- Unit_Price: double (nullable = true)
 |-- Extended_Amount: double (nullable = true)
 |-- Product_Standard_Cost: double (nullable = true)
 |-- Total_Product_Cost: double (nullable = true)
 |-- Sales_Amount: double (nullable = true)



sum_Unit_Price,sum_Extended_Amount,sum_Product_Standard_Cost,sum_Total_Product_Cost,sum_Sales_Amount,nulls_Unit_Price,nulls_Extended_Amount,nulls_Product_Standard_Cost,nulls_Total_Product_Cost,nulls_Sales_Amount
5.640443448007946E7,1.1033678150975268E8,4.397171650999333E7,9.725798807005282E7,1.098092739997523E8,0,0,0,0,0


## Handle Missing or Null Values (DimCustomer & FactSales)

Profiling findings:
- **DimCustomer**: no nulls or placeholder values in any column (defensive `Unknown` fill applied anyway)
- **FactSales**: only `ShipDateKey` has nulls (2,113 rows = orders not yet shipped; all are the latest orders dated 2020-06-15). Nulls are replaced with `0` sentinel = "not shipped yet" so rows are not lost in aggregations.

In [0]:
# --- DimCustomer: fill any missing text values with 'Unknown' (defensive) ---
text_cols = [c for c, dt in dim_cust.dtypes if dt == "string"]
dim_cust = dim_cust.fillna("Unknown", subset=text_cols)

# --- FactSales: ShipDateKey is null for orders not yet shipped ---
# All 2,113 null rows are the most recent orders (OrderDateKey = 20200615).
# Use 0 as a sentinel meaning 'not shipped yet' instead of dropping the rows,
# so the newest sales still count in aggregations.
fact_sales = fact_sales.fillna(0, subset=["ShipDateKey"])

# Defensive: measures must never be null after casting
fact_sales = fact_sales.fillna(0.0, subset=money_cols)

# --- Verify: no remaining nulls in either table, row counts unchanged ---
def null_total(df, name):
    total = df.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    print(f"{name}: {df.count():,} rows, total nulls = {sum(v for v in total.asDict().values())}")

null_total(dim_cust, "dim_cust")
null_total(fact_sales, "fact_sales")

# Confirm the unshipped orders were kept (sentinel 0), not dropped
print(f"Rows with ShipDateKey = 0 (not yet shipped): {fact_sales.filter(F.col('ShipDateKey') == 0).count():,}")

dim_cust: 18,485 rows, total nulls = 0
fact_sales: 121,253 rows, total nulls = 0
Rows with ShipDateKey = 0 (not yet shipped): 2,113


Check for empty customer names

In [0]:
# --- Step 4: Check if any customer names are empty ---
# An 'empty' name means: NULL, zero-length string, or whitespace-only string
empty_names = dim_cust.filter(
    F.col("Customer").isNull() | (F.trim(F.col("Customer")) == "")
)

empty_cnt = empty_names.count()
total_cnt = dim_cust.count()
print(f"Customers with null/empty names: {empty_cnt:,} of {total_cnt:,}")

if empty_cnt > 0:
    display(empty_names.select("CustomerKey", "Customer"))
else:
    print("All customer names are populated - no empty values found.")

# Sample of names to eyeball
print("Sample customer names:")
dim_cust.select("Customer").show(5, truncate=False)

# Also check for placeholder-style names like '[Not Applicable]'
# (not technically empty, but no real customer name either)
placeholder_names = dim_cust.filter(F.col("Customer").rlike(r"^\[.*\]$"))
print(f"Customers with bracketed placeholder names: {placeholder_names.count():,}")
if placeholder_names.count() > 0:
    display(placeholder_names)

Customers with null/empty names: 0 of 18,485
All customer names are populated - no empty values found.
Sample customer names:
+----------------+
|Customer        |
+----------------+
|[Not Applicable]|
|Jon Yang        |
|Eugene Huang    |
|Ruben Torres    |
|Christy Zhu     |
+----------------+
only showing top 5 rows
Customers with bracketed placeholder names: 1


CustomerKey,Customer_ID,Customer,City,State-Province,Country-Region,Postal_Code
-1,[Not Applicable],[Not Applicable],[Not Applicable],[Not Applicable],[Not Applicable],[Not Applicable]


## Step 5: Joining Datasets

Five joins onto FactSales (SalesData), then combined into one dataframe:

1. **Regional insights**: FactSalesOrderDetails (SalesOrderLineKey) + FactSalesTerritoryDetails (SalesTerritoryKey)
2. **Customer purchases**: DimCustomer (CustomerKey)
3. **Reseller performance**: DimReseller (ResellerKey)
4. **Product-wise sales**: DimProduct (ProductKey)
5. **Trends over time**: DimDate (OrderDateKey = DateKey)

Referential integrity verified beforehand: 0 unmatched keys for every join. The combined result is written to `workspace.silver.FactSalesCombined` (Delta).

In [0]:
# --- Step 5: Joining Datasets ---

# DimCustomer and DimReseller both carry City/State-Province/Country-Region/Postal Code.
# Prefix them so the combined dataframe has no ambiguous column names.
# NOTE: 'State-Province' and 'Country-Region' keep their hyphens (the column-name
# cleaner only replaces spaces), so use the exact cleaned names here.
geo_cols = ["City", "State-Province", "Country-Region", "Postal_Code"]
dim_cust_j     = dim_cust.withColumnsRenamed({c: f"Customer_{c}" for c in geo_cols})
dim_reseller_j = dim_reseller.withColumnsRenamed({c: f"Reseller_{c}" for c in geo_cols})

# 1) Regional insights: order details + sales territory, via FactSales keys
sales_region = (
    fact_sales
    .join(fact_order, "SalesOrderLineKey")   # Channel, Sales_Order, Sales_Order_Line
    .join(fact_terr, "SalesTerritoryKey")      # Region, Country, Group
)

# 2) Customer purchases
sales_customer = fact_sales.join(dim_cust_j, "CustomerKey")

# 3) Reseller performance
sales_reseller = fact_sales.join(dim_reseller_j, "ResellerKey")

# 4) Product-wise sales
sales_product = fact_sales.join(dim_product, "ProductKey")

# 5) Trends over time (OrderDateKey -> DateKey)
sales_date = fact_sales.join(dim_date, F.col("OrderDateKey") == F.col("DateKey"))

for df, name in [(sales_region, "sales_region"), (sales_customer, "sales_customer"),
                 (sales_reseller, "sales_reseller"), (sales_product, "sales_product"),
                 (sales_date, "sales_date")]:
    print(f"{name:<16}{df.count():>10,} rows | columns: {len(df.columns)}")

# --- Combine all five into a single dataframe ---
fact_sales_combined = (
    fact_sales
    .join(fact_order, "SalesOrderLineKey")
    .join(fact_terr, "SalesTerritoryKey")
    .join(dim_cust_j, "CustomerKey")
    .join(dim_reseller_j, "ResellerKey")
    .join(dim_product, "ProductKey")
    .join(dim_date, F.col("OrderDateKey") == F.col("DateKey"))
)

print(f"\nfact_sales_combined: {fact_sales_combined.count():,} rows | columns: {len(fact_sales_combined.columns)}")

sales_region       121,253 rows | columns: 21
sales_customer     121,253 rows | columns: 21
sales_reseller     121,253 rows | columns: 22
sales_product      121,253 rows | columns: 23
sales_date         121,253 rows | columns: 22

fact_sales_combined: 121,253 rows | columns: 49


In [0]:
# --- Write the combined dataset to the Silver layer as a Delta table ---
fact_sales_combined.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.FactSalesCombined")

# --- Verify: row count must match FactSales exactly (no fan-out, no drops) ---
combined_tbl = spark.table(f"{SILVER}.FactSalesCombined")
print(f"FactSales rows:       {fact_sales.count():,}")
print(f"FactSalesCombined:    {combined_tbl.count():,} rows | columns: {len(combined_tbl.columns)}")
assert combined_tbl.count() == fact_sales.count(), "Row count mismatch - join fan-out detected!"
print("Row counts match - no fan-out, no dropped rows.")

# Quick sanity peek at the joined result
display(combined_tbl.select("SalesOrderLineKey", "Channel", "Region", "Country",
                            "Customer", "Reseller", "Product", "Category",
                            "Date", "Sales_Amount"))

FactSales rows:       121,253
FactSalesCombined:    121,253 rows | columns: 49
Row counts match - no fan-out, no dropped rows.


SalesOrderLineKey,Channel,Region,Country,Customer,Reseller,Product,Category,Date,Sales_Amount
43659001,Reseller,Southeast,United States,[Not Applicable],Better Bike Shop,"Mountain-100 Black, 42",Bikes,2017-07-02,2024.99
43659002,Reseller,Southeast,United States,[Not Applicable],Better Bike Shop,"Mountain-100 Black, 44",Bikes,2017-07-02,6074.98
43659003,Reseller,Southeast,United States,[Not Applicable],Better Bike Shop,"Mountain-100 Black, 48",Bikes,2017-07-02,2024.99
43659004,Reseller,Southeast,United States,[Not Applicable],Better Bike Shop,"Mountain-100 Silver, 38",Bikes,2017-07-02,2039.99
43659005,Reseller,Southeast,United States,[Not Applicable],Better Bike Shop,"Mountain-100 Silver, 42",Bikes,2017-07-02,2039.99
43659006,Reseller,Southeast,United States,[Not Applicable],Better Bike Shop,"Mountain-100 Silver, 44",Bikes,2017-07-02,4079.99
43659007,Reseller,Southeast,United States,[Not Applicable],Better Bike Shop,"Mountain-100 Silver, 48",Bikes,2017-07-02,2039.99
43659008,Reseller,Southeast,United States,[Not Applicable],Better Bike Shop,"Long-Sleeve Logo Jersey, M",Clothing,2017-07-02,86.52
43659009,Reseller,Southeast,United States,[Not Applicable],Better Bike Shop,"Long-Sleeve Logo Jersey, XL",Clothing,2017-07-02,28.84
43659010,Reseller,Southeast,United States,[Not Applicable],Better Bike Shop,"Mountain Bike Socks, M",Clothing,2017-07-02,34.2


## Feature Engineering

1. **Total Revenue per Customer** from `FactSalesCombined` using `groupBy()` + `sum()`
2. **Customer Segmentation** based on purchase behavior:
   - `Total_Revenue > 50000` → **High-Value**
   - `Total_Revenue > 20000` → **Medium-Value**
   - else → **Low-Value**

Result is written to `workspace.silver.FactCustomerSegment` (Delta).

In [0]:
# --- Feature Engineering: Total Revenue per Customer ---
combined_tbl = spark.table(f"{SILVER}.FactSalesCombined")
customer_revenue = (
    combined_tbl
    .groupBy("CustomerKey", "Customer")
    .agg(
        F.sum("Sales_Amount").alias("Total_Revenue"),
        F.countDistinct("SalesOrderLineKey").alias("Total_Orders"),
        F.sum("Order_Quantity").alias("Total_Quantity"),
    )
)

print(f"Customers with purchases: {customer_revenue.count():,}")
display(customer_revenue.orderBy(F.desc("Total_Revenue")).limit(10))

# --- Customer Segmentation based on purchase behavior ---
fact_customer_segment = customer_revenue.withColumn(
    "Customer_Segment",
    F.when(F.col("Total_Revenue") > 50000, "High-Value")
     .when(F.col("Total_Revenue") > 20000, "Medium-Value")
     .otherwise("Low-Value"),
)

fact_customer_segment.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.FactCustomerSegment")

# --- Verify ---
segment_tbl = spark.table(f"{SILVER}.FactCustomerSegment")
seg_dist = segment_tbl.groupBy("Customer_Segment").agg(
    F.count("*").alias("customers"),
    F.sum("Total_Revenue").alias("revenue"),
)
display(seg_dist)

assert segment_tbl.count() == customer_revenue.count(), "Segment table lost/gained rows!"
assert seg_dist.agg(F.sum("customers")).collect()[0][0] == customer_revenue.count(), "Segment distribution does not sum to customer count!"
print(f"FactCustomerSegment: {segment_tbl.count():,} rows - verified")

Customers with purchases: 18,485


CustomerKey,Customer,Total_Revenue,Total_Orders,Total_Quantity
-1,[Not Applicable],8.045059610998881E7,60855,214378
12301,Nichole Nara,13295.38,13,13
12132,Kaitlyn Henderson,13294.269999999999,14,14
12308,Margaret He,13269.269999999999,14,14
12131,Randall Dominguez,13265.99,11,11
12300,Adriana Gonzalez,13242.699999999999,10,10
12321,Rosa Hu,13215.649999999998,15,15
12124,Brandi Gill,13195.64,12,12
12307,Brad She,13173.189999999999,11,11
12296,Francisco Sara,13164.639999999998,12,12


Customer_Segment,customers,revenue
Low-Value,18484,2.935867788999971E7
High-Value,1,8.045059610998881E7


FactCustomerSegment: 18,485 rows - verified


##   Store Transformed Data in the Silver Layer

- `FactSalesCombined` and `FactCustomerSegment` stored as Silver Delta tables
- `FactSalesCombined` **partitioned by `SalesTerritoryKey`** for optimized queries
- Validation: row counts (FactSalesCombined = 121,253; FactCustomerSegment = 18,485) + business logic tests

In [0]:
# --- Re-write FactSalesCombined partitioned by Sales Territory ---
# FactCustomerSegment is already stored as a Silver Delta table (previous step).
combined_tbl = spark.table(f"{SILVER}.FactSalesCombined")

combined_tbl.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("SalesTerritoryKey") \
    .saveAsTable(f"{SILVER}.FactSalesCombined")

# Verify the table is now partitioned by SalesTerritoryKey
print(spark.sql(f"DESCRIBE TABLE {SILVER}.FactSalesCombined").collect()[-1])
partitions = spark.sql(f"SHOW PARTITIONS {SILVER}.FactSalesCombined")
print(f"Partitions: {partitions.count()}")
display(partitions)

Row(col_name='# col_name', data_type='data_type', comment='comment')
Partitions: 10


SalesTerritoryKey
10
1
7
3
6
8
5
4
9
2


In [0]:
# --- Validate transformed data: row counts ---
sales_combined_df = spark.table(f"{SILVER}.FactSalesCombined")
customer_segments = spark.table(f"{SILVER}.FactCustomerSegment")

row_count = sales_combined_df.count()
print(f"Total Records in Sales Combined Data: {row_count}")

row_count = customer_segments.count()
print(f"Total Records in Customer Segmentation Data: {row_count}")

# --- Business logic tests ---
# 1) Revenue in the segment table must reconcile with combined sales
total_sales = sales_combined_df.agg(F.sum("Sales_Amount")).collect()[0][0]
total_revenue = customer_segments.agg(F.sum("Total_Revenue")).collect()[0][0]
assert abs(total_sales - total_revenue) < 0.01, "Segment revenue does not reconcile with combined sales!"
print(f"Revenue reconciles: combined sales = segment revenue = {total_sales:,.2f}")

# 2) Segment labels must respect the thresholds (High > 50000, Medium > 20000, else Low)
bad_segments = customer_segments.filter(
    ((F.col("Customer_Segment") == "High-Value")   & (F.col("Total_Revenue") <= 50000)) |
    ((F.col("Customer_Segment") == "Medium-Value") & (F.col("Total_Revenue") <= 20000)) |
    ((F.col("Customer_Segment") == "Low-Value")    & (F.col("Total_Revenue") > 20000))
).count()
assert bad_segments == 0, f"{bad_segments} rows violate the segmentation thresholds!"
print("All segment labels respect the revenue thresholds")

# 3) No null keys in the combined fact table
null_keys = sales_combined_df.filter(
    F.col("SalesOrderLineKey").isNull() | F.col("SalesTerritoryKey").isNull()
    | F.col("CustomerKey").isNull() | F.col("ProductKey").isNull()
).count()
assert null_keys == 0, f"{null_keys} rows have null keys!"
print("No null keys in FactSalesCombined")

Total Records in Sales Combined Data: 121253
Total Records in Customer Segmentation Data: 18485
Revenue reconciles: combined sales = segment revenue = 109,809,274.00
All segment labels respect the revenue thresholds
No null keys in FactSalesCombined


In [0]:
row_count = sales_combined_df.count() 

print(f"Total Records in Sales Combined Data: {row_count}") 


row_count = customer_segments.count() 

print(f"Total Records in Customer Segmentation Data: {row_count}") 

 
current_username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
current_username

Total Records in Sales Combined Data: 121253
Total Records in Customer Segmentation Data: 18485


'hemanth.appari@maqsoftware.com'